# Train DocVerify's English AI-text detector (free, no inference API key)

Upload the DocVerify project ZIP, stream a balanced subset of the public `artem9k/ai-text-detection-pile` dataset, evaluate the model on a held-out split, then download `ai_detector.joblib`. Training runs in this Colab session and may take several minutes. The resulting score is experimental, not proof of authorship.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Select the DocVerify project ZIP
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Please upload the DocVerify project ZIP file.')


In [ ]:
import os, glob, zipfile, sys, subprocess
workdir = '/content/docverify_training'
os.makedirs(workdir, exist_ok=True)
with zipfile.ZipFile(zip_name) as archive:
    archive.extractall(workdir)
scripts = glob.glob(os.path.join(workdir, '**', 'train_ai_detector.py'), recursive=True)
if not scripts:
    raise FileNotFoundError('train_ai_detector.py was not found inside the ZIP.')
project_dir = os.path.dirname(scripts[0])
os.chdir(project_dir)
print('Project directory:', project_dir)


In [ ]:
%pip -q install -r requirements-train.txt


## Train and evaluate

Default uses up to 12,000 human and 12,000 AI samples. For a quicker first run, change `12000` to `5000`. The dataset is streamed; the entire ~2 GB dataset is not downloaded.

In [ ]:
result = subprocess.run([sys.executable, 'train_ai_detector.py', '--max-per-class', '12000'], text=True)
if result.returncode != 0:
    raise RuntimeError('Training failed. Read the error output above and retry.')
model_path = os.path.join(project_dir, 'models', 'ai_detector.joblib')
metrics_path = os.path.join(project_dir, 'models', 'evaluation.json')
print('Model:', model_path, os.path.getsize(model_path) / (1024 * 1024), 'MB')
print('Metrics report:', metrics_path)


## Download the model artifact

Place this downloaded file in your local DocVerify project at exactly `models/ai_detector.joblib`, then commit it and redeploy Vercel. Do not upload the training dataset itself.

In [ ]:
from google.colab import files
files.download(model_path)
